<a href="https://www.kaggle.com/code/lalit7881/smartphone-addiction-eda?scriptVersionId=347408880" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
# Suppress warnings for a cleaner output
import warnings
warnings.filterwarnings('ignore')

# Ensure inline plotting and set backend for matplotlib
%matplotlib inline
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.switch_backend('Agg')
import seaborn as sns
from pathlib import Path
import os

# Set a seaborn style for all of our plots
sns.set(style="whitegrid")
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/anthonytherrien/predicting-smartphone-addiction-vault/submission (1).csv
/kaggle/input/datasets/anthonytherrien/predicting-smartphone-addiction-vault/submission.csv


## Loading dataset

In [2]:
## Helper function to locate our data files dynamically
DATA_ROOTS = [
    Path('/kaggle/input'),
    Path('/kaggle/input'),
    Path('/kaggle/input/datasets')
]

def resolve_data_file(file_name=None, suffixes=None):
    """
    Helper function to resolve the data file path based on file name or file suffixes.
    """
    suffixes = tuple(suffixes or [])
    for root in DATA_ROOTS:
        if not root.exists():
            continue
        matches = [p for p in root.rglob('*') if p.is_file()]
        if file_name:
            exact = [p for p in matches if p.name == file_name]
            if exact:
                return exact[0]
        if suffixes:
            typed = [p for p in matches if p.suffix.lower() in suffixes]
            if typed:
                return typed[0]
    raise FileNotFoundError(f'Could not resolve data file: {file_name or suffixes}')

# Example usage:
# file_path = resolve_data_file(file_name='submission.csv')
# df = pd.read_csv(file_path)

In [3]:
## Load the first submission file
try:
    file1_path = resolve_data_file(file_name='submission (1).csv')
    df1 = pd.read_csv(file1_path, encoding='ascii', delimiter=',')
    print(f'Loaded file: {file1_path}')
except FileNotFoundError as e:
    print(e)

## Load the second submission file
try:
    file2_path = resolve_data_file(file_name='submission.csv')
    df2 = pd.read_csv(file2_path, encoding='ascii', delimiter=',')
    print(f'Loaded file: {file2_path}')
except FileNotFoundError as e:
    print(e)

# For further analysis, we will use the first file if both are available,
# but you might want to compare the two files if they differ.
df = df1.copy() if 'df1' in globals() else (df2.copy() if 'df2' in globals() else None)

if df is not None:
    print('Preview of the dataframe:')
    display(df.head())
else:
    print('No data loaded. Please check your file paths.')

Loaded file: /kaggle/input/datasets/anthonytherrien/predicting-smartphone-addiction-vault/submission (1).csv
Loaded file: /kaggle/input/datasets/anthonytherrien/predicting-smartphone-addiction-vault/submission.csv
Preview of the dataframe:


,id,addicted_label
0,691369,0.790607
1,691370,0.476293
2,691371,0.379679
3,691372,0.625755
4,691373,0.684513


## Data cleaning


In [4]:
if df is not None:
    # Display basic information about the dataframe
    print("DataFrame Info:")
    display(df.info())
    
    # Check for duplicate rows
    duplicate_count = df.duplicated().sum()
    print(f'Number of duplicate rows: {duplicate_count}')

    # Check for missing values
    missing_values = df.isnull().sum()
    print('Missing values in each column:')
    display(missing_values)

    # Convert data types if necessary
    df['id'] = pd.to_numeric(df['id'], errors='coerce')
    df['addicted_label'] = pd.to_numeric(df['addicted_label'], errors='coerce')

    # Summary statistics
    print('Summary Statistics:')
    display(df.describe())
else:
    print('Dataframe is not loaded. Skipping cleaning steps.')

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 296302 entries, 0 to 296301
Data columns (total 2 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              296302 non-null  int64  
 1   addicted_label  296302 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 4.5 MB


None

Number of duplicate rows: 0
Missing values in each column:


id                0
addicted_label    0
dtype: int64

Summary Statistics:


,id,addicted_label
count,296302.000000,296302.000000
mean,839519.500000,0.500000
std,85535.164068,0.288676
min,691369.000000,0.000002
25%,765444.250000,0.250001
50%,839519.500000,0.500000
75%,913594.750000,0.749999
max,987670.000000,0.999998


## EDA

In [5]:
if df is not None:
    # Basic histogram for addicted_label
    plt.figure(figsize=(8, 4))
    sns.histplot(df['addicted_label'], kde=True, bins=20)
    plt.title('Distribution of Addicted Label')
    plt.xlabel('Addicted Label')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    # Box Plot for the addicted_label
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df['addicted_label'])
    plt.title('Box Plot of Addicted Label')
    plt.tight_layout()
    plt.show()

    # Pair Plot to understand any relationships, though limited with only two columns
    sns.pairplot(df)
    plt.suptitle('Pair Plot of the Dataset', y=1.02)
    plt.show()

    # If the addicted_label seems to be binary, a count plot might be more appropriate
    if set(df['addicted_label'].unique()).issubset({0, 1}):
        plt.figure(figsize=(6, 4))
        sns.countplot(x=df['addicted_label'])
        plt.title('Count Plot of Addicted Label')
        plt.tight_layout()
        plt.show()
else:
    print('Dataframe is not loaded. Skipping EDA steps.')

In [6]:
df.columns

Index(['id', 'addicted_label'], dtype='object')

## Thank you...pls upvote!!!!!!!!!